# Stage 00 · Vulkan GPU Puzzles — Puzzle 02: Zip (`out[i] = a[i] + b[i]`)

In **Puzzle 01**, our kernel took **1 input buffer** (`a`) and wrote **1 output buffer** (`out`). In **Puzzle 02 (`Zip`)**, we combine **two GPU arrays element-by-element in parallel**.

### What's New in Puzzle 02?

| Concept | Puzzle 01 (`Map`) | Puzzle 02 (`Zip`) |
|---|---|---|
| **Vulkan Storage Buffers** | 2 bindings (`out` = 0, `a` = 1) | **3 bindings** (`out` = 0, `a` = 1, `b` = 2) |
| **Memory Traffic / Element** | 8 bytes (1 read + 1 write) | **12 bytes** (2 reads + 1 write) |
| **Arithmetic Intensity** | 1 FLOP / 8 bytes = `0.125` | 1 FLOP / 12 bytes = `0.083` (**memory-bandwidth bound!**) |
| **AI Connection** | Adding a bias scalar | **Residual skip connections** (`x + residual`) in ResNets & Transformers! |

**Goal:** Given two GPU arrays `a` and `b` of length `n`, compute `out[i] = a[i] + b[i]` for every `0 <= i < n`.

In [1]:
import sys, time
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vknumpy").is_dir()), Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import vknumpy as vnp
from vknumpy import global_id_x

@vnp.kernel(local_size=(64, 1, 1))
def puzzle_02_zip(out, a, b, n: int):
    i = global_id_x()
    if i < n:
        out[i] = a[i] + b[i]

print(f"Compiled `{puzzle_02_zip.name}` -> {len(puzzle_02_zip.spirv_bytes):,} bytes of SPIR-V bytecode!")
print("\n=== Generated Vulkan GLSL 450 (notice bindings 0, 1, 2!) ===")
print(puzzle_02_zip.glsl)

Compiled `puzzle_02_zip` -> 1,576 bytes of SPIR-V bytecode!

=== Generated Vulkan GLSL 450 (notice bindings 0, 1, 2!) ===
#version 450
layout(local_size_x = 64, local_size_y = 1, local_size_z = 1) in;

layout(set = 0, binding = 0) buffer Buf__out { float _out[]; };
layout(set = 0, binding = 1) buffer Buf_a { float a[]; };
layout(set = 0, binding = 2) buffer Buf_b { float b[]; };

layout(push_constant) uniform PC {
    int n;
} pc;

void main() {
    int i = int(gl_GlobalInvocationID.x);
    if (((i < pc.n))) {
        _out[i] = (a[i] + b[i]);
    }
}



### Run the Auto-Grader & Benchmark on 1,000,000 Elements

Notice we print the original `a`, original `b`, and the GPU result `a + b` side by side, plus effective memory bandwidth (`3 * N * 4` bytes = 2 reads + 1 write).

In [2]:
def check_puzzle_02(kernel_fn):
    for n in (1, 17, 64, 1000, 10_000):
        a_np = np.arange(n, dtype=np.float32)
        b_np = np.arange(n, dtype=np.float32) * 2.0 + 5.0
        a, b = vnp.array(a_np), vnp.array(b_np)
        out = vnp.zeros((n,), dtype=np.float32)
        kernel_fn[(n + 63) // 64](out, a, b, n)
        np.testing.assert_allclose(out.to_numpy(), a_np + b_np, atol=1e-5)
    print("PASS! Puzzle 02 (Zip) passed all 5 test sizes (1, 17, 64, 1000, 10000)!")

check_puzzle_02(puzzle_02_zip)

N = 1_000_000
a_gpu = vnp.array(np.arange(N, dtype=np.float32))
b_gpu = vnp.array(np.arange(N, dtype=np.float32) * 2.0 + 5.0)
out_gpu = vnp.zeros((N,), dtype=np.float32)
grid = (N + 63) // 64

puzzle_02_zip[grid](out_gpu, a_gpu, b_gpu, N)  # warmup
t0 = time.perf_counter()
for _ in range(50):
    puzzle_02_zip[grid](out_gpu, a_gpu, b_gpu, N)
ms = (time.perf_counter() - t0) * 1000.0 / 50.0
gb_s = (3 * N * 4) / (ms * 1e-3) / 1e9

print(f"1,000,000 elements on Vulkan GPU: {ms:.2f} ms/launch (~{gb_s:.1f} GB/s effective bandwidth)")
print("Original a (first 6):", a_gpu.to_numpy()[:6])
print("Original b (first 6):", b_gpu.to_numpy()[:6])
print("a + b      (first 6):", out_gpu.to_numpy()[:6])

PASS! Puzzle 02 (Zip) passed all 5 test sizes (1, 17, 64, 1000, 10000)!
1,000,000 elements on Vulkan GPU: 0.30 ms/launch (~39.4 GB/s effective bandwidth)
Original a (first 6): [0. 1. 2. 3. 4. 5.]
Original b (first 6): [ 5.  7.  9. 11. 13. 15.]
a + b      (first 6): [ 5.  8. 11. 14. 17. 20.]


ggml_vulkan: Found 1 Vulkan devices:
ggml_vulkan: 0 = AMD Custom GPU 0405 (RADV VANGOGH) (radv) | uma: 1 | rebar: 1 | fp16: 0 | bf16: 0 | fp4: 0 | warp size: 32 | shared memory: 65536 | int dot: 1 | matrix cores: none


---
### Bonus: Fused AXPY (`out[i] = alpha * a[i] + b[i]`) — Adding a `float` Push Constant

Because elementwise kernels are **memory-bandwidth bound**, fusing a scalar multiply (`alpha * a[i]`) into the same pass costs **almost zero extra time** — this is the exact operation behind `SGD` weight updates (`w = w - lr * grad`) in `vktorch`!

In [3]:
@vnp.kernel(local_size=(64, 1, 1))
def fused_axpy(out, a, b, n: int, alpha: float):
    i = global_id_x()
    if i < n:
        out[i] = alpha * a[i] + b[i]

alpha = 0.5
fused_axpy[grid](out_gpu, a_gpu, b_gpu, N, alpha)  # warmup
t0 = time.perf_counter()
for _ in range(50):
    fused_axpy[grid](out_gpu, a_gpu, b_gpu, N, alpha)
ms_axpy = (time.perf_counter() - t0) * 1000.0 / 50.0

np.testing.assert_allclose(out_gpu.to_numpy()[:6], alpha * a_gpu.to_numpy()[:6] + b_gpu.to_numpy()[:6], atol=1e-5)
print(f"Fused AXPY (alpha={alpha}): {ms_axpy:.2f} ms/launch (virtually identical to plain Zip!)")
print("Original a     (first 6):", a_gpu.to_numpy()[:6])
print("Original b     (first 6):", b_gpu.to_numpy()[:6])
print("0.5 * a + b    (first 6):", out_gpu.to_numpy()[:6])

Fused AXPY (alpha=0.5): 0.31 ms/launch (virtually identical to plain Zip!)
Original a     (first 6): [0. 1. 2. 3. 4. 5.]
Original b     (first 6): [ 5.  7.  9. 11. 13. 15.]
0.5 * a + b    (first 6): [ 5.   7.5 10.  12.5 15.  17.5]


---
## Complete Standalone Script (All-in-One)

In [4]:
import sys, time
from pathlib import Path

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vknumpy").is_dir()), Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import vknumpy as vnp
from vknumpy import global_id_x


@vnp.kernel(local_size=(64, 1, 1))
def zip_add(out, a, b, n: int):
    i = global_id_x()
    if i < n:
        out[i] = a[i] + b[i]


N = 1_000_000
a_np = np.arange(N, dtype=np.float32)
b_np = np.arange(N, dtype=np.float32) * 2.0 + 5.0
a_gpu, b_gpu = vnp.array(a_np), vnp.array(b_np)
out_gpu = vnp.zeros((N,), dtype=np.float32)
grid = (N + 63) // 64

zip_add[grid](out_gpu, a_gpu, b_gpu, N)  # warmup
t0 = time.perf_counter()
for _ in range(50):
    zip_add[grid](out_gpu, a_gpu, b_gpu, N)
ms = (time.perf_counter() - t0) * 1000.0 / 50.0
np.testing.assert_allclose(out_gpu.to_numpy()[:100], a_np[:100] + b_np[:100], atol=1e-5)

print(f"PASS! {N:,} elements in {ms:.2f} ms")
print("Original a (first 6):", a_np[:6])
print("Original b (first 6):", b_np[:6])
print("a + b      (first 6):", out_gpu.to_numpy()[:6])

PASS! 1,000,000 elements in 0.30 ms
Original a (first 6): [0. 1. 2. 3. 4. 5.]
Original b (first 6): [ 5.  7.  9. 11. 13. 15.]
a + b      (first 6): [ 5.  8. 11. 14. 17. 20.]
